# Kaze on Colab

GPU は T4 (16GB) でも A100 でも動きます。上から順に実行してください。
T4 は sm_75 なので、bf16 ではなく fp16 を使います（既定が fp16 です）。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q torch triton transformers safetensors fastapi uvicorn
import torch, triton
print(torch.__version__, triton.__version__, torch.cuda.get_device_capability(0))

## コードを用意

リポジトリを持っている場合は clone、そうでなければ kaze/ ディレクトリをアップロードしてください。

In [ ]:
import os
if not os.path.isdir('kaze'):
    # 自分のリポジトリに置き換える
    !git clone -q https://github.com/you/kaze.git repo && cp -r repo/kaze . && cp -r repo/bench .
import kaze; print('kaze', kaze.__version__)

## モデルを読む（int8, group 128）

In [ ]:
from kaze.llm import LLM
from kaze.sampler import SamplingParams

llm = LLM('Qwen/Qwen2.5-0.5B-Instruct', kind='int8', group=128, lm_head_kind='int8',
          max_batch=32, max_batched_tokens=2048)

In [ ]:
out = llm.chat([{'role': 'user', 'content': 'GPU の俳句を一つ'}],
               SamplingParams(max_tokens=48, temperature=0.0))
print(out.text)
print(f"ttft {out.ttft_ms:.0f} ms, tpot {out.tpot_ms:.2f} ms")

## ベンチ

In [ ]:
from bench.run import bench_prefill, bench_decode, bench_concurrency
print('prefill', {k: round(v,1) for k,v in bench_prefill(llm, lengths=(128,512)).items()})
print('decode ', {k: round(v,2) for k,v in bench_decode(llm, 64, 128).items()})
print('conc   ', {n: round(r['agg_tok_s'],1) for n,r in bench_concurrency(llm, levels=(1,8,32)).items()})

## サーバとして動かす

Colab では localhost に立つので、`/v1/chat/completions` を curl で叩けます。

In [ ]:
import threading, time, requests, json
from kaze.server import create_app
import uvicorn

app = create_app(llm)
threading.Thread(target=lambda: uvicorn.run(app, host='127.0.0.1', port=8000, log_level='warning'), daemon=True).start()
time.sleep(3)
r = requests.post('http://127.0.0.1:8000/v1/chat/completions', json={
    'messages': [{'role': 'user', 'content': '1 から 5 まで数えて'}],
    'max_tokens': 32, 'temperature': 0})
print(r.json()['choices'][0]['message']['content'])